# CalmFruits — неделя 2: baseline и протокол оценки

Тетрадь фиксирует query-level development/validation split, оценивает два готовых поиска по одному каталогу и сохраняет результаты. Официальная test-разметка не читается.

In [1]:
from __future__ import annotations

import importlib.metadata
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import load
from scipy import sparse
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
assert (ROOT / 'src' / 'calmfruits').exists()
sys.path.insert(0, str(ROOT / 'src'))

from calmfruits.catalog import clean_query
from calmfruits.data import load_local_env, read_parquet_from_s3
from calmfruits.evaluation import K_VALUES, RELEVANCE_THRESHOLD, build_golden_set, evaluate_search, split_queries, summarize_metrics
from calmfruits.search import LexicalSearch, SemanticSearch, SEMANTIC_MODEL, SEMANTIC_REVISION

SEED = 42
np.random.seed(SEED)
CATALOG_DIR = ROOT / 'artifacts' / 'catalog'
INDEX_DIR = ROOT / 'artifacts' / 'indexes'
EVALUATION_DIR = ROOT / 'artifacts' / 'evaluation'
REPORT_DIR = ROOT / 'reports'
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
assert load_local_env(ROOT / '.env'), 'S3 credentials are required in local .env'
environment = pd.DataFrame({'value': {
    'python': sys.version.split()[0], 'pandas': importlib.metadata.version('pandas'),
    'scikit_learn': importlib.metadata.version('scikit-learn'), 'sentence_transformers': importlib.metadata.version('sentence-transformers'),
    'seed': SEED, 'device': 'CPU', 'K_values': K_VALUES, 'relevance_threshold': RELEVANCE_THRESHOLD,
    'sources': 'prepared evaluation catalog + indexes + queries_synthetic_train.parquet',
}})
display(environment)


,value
python,3.12.6
pandas,2.2.3
scikit_learn,1.6.1
sentence_transformers,5.5.1
seed,42
device,CPU
K_values,"(1, 3, 5, 10)"
relevance_threshold,2
sources,prepared evaluation catalog + indexes + querie...


## 1. Загрузка фиксированного каталога и индексов

Каталог и оба индекса созданы в тетради 02. Проверяем порядок `imt_id` по манифесту до любой оценки.

In [2]:
evaluation_catalog = pd.read_parquet(CATALOG_DIR / 'evaluation_catalog.parquet')
queries_train = read_parquet_from_s3('queries_synthetic_train.parquet')
manifest = json.loads((INDEX_DIR / 'index_manifest.json').read_text())
assert evaluation_catalog.imt_id.astype(int).tolist() == manifest['catalog']['imt_id_order']
lexical = LexicalSearch(evaluation_catalog, load(INDEX_DIR / 'tfidf_vectorizer.joblib'), sparse.load_npz(INDEX_DIR / 'tfidf_matrix.npz'))
semantic = SemanticSearch(evaluation_catalog, SentenceTransformer(SEMANTIC_MODEL, revision=SEMANTIC_REVISION, device='cpu'), np.load(INDEX_DIR / 'semantic_embeddings.npy'))
assert lexical.matrix.shape[0] == semantic.embeddings.shape[0] == len(evaluation_catalog)
assert evaluation_catalog.imt_id.is_unique
display(pd.DataFrame({'catalog_size': [len(evaluation_catalog)], 'tfidf_shape': [str(lexical.matrix.shape)], 'semantic_shape': [str(semantic.embeddings.shape)]}))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,catalog_size,tfidf_shape,semantic_shape
0,3280,"(3280, 50000)","(3280, 384)"


**Вывод по загрузке каталога и индексов.** Манифест подтвердил одинаковый порядок 3 280 `imt_id` для каталога, TF-IDF-матрицы и embedding-матрицы 3 280×384; оба поиска оцениваются на одном наборе карточек.

## 2. Query-level split и golden-set

Разбиение использует нормализованный текст запроса как группу: одинаковый текст не попадает одновременно в development и validation.

In [3]:
assert queries_train.groupby('query_id').query_text.nunique().eq(1).all()
query_split = split_queries(queries_train, seed=SEED)
golden_validation = build_golden_set(queries_train, query_split)
assert set(query_split.loc[query_split['split'].eq('development'), 'normalized_query_text']).isdisjoint(set(query_split.loc[query_split['split'].eq('validation'), 'normalized_query_text']))
assert query_split.query_id.nunique() == queries_train.query_id.nunique()
assert golden_validation.query_id.nunique() == query_split['split'].eq('validation').sum()
query_split.to_csv(EVALUATION_DIR / 'query_split.csv', index=False)
golden_validation.to_parquet(EVALUATION_DIR / 'golden_validation.parquet', index=False)
split_summary = query_split['split'].value_counts().rename_axis('split').reset_index(name='queries')
relevance_by_split = golden_validation.relevance.value_counts().sort_index().rename_axis('relevance').reset_index(name='pairs')
display(split_summary)
display(relevance_by_split)


,split,queries
0,development,560
1,validation,140


,relevance,pairs
0,1,1960
1,2,181
2,3,140


**Вывод по `query_split` и `golden_validation`.** Group-aware split дал 560 development- и 140 validation-запросов, без пересечения нормализованных текстов. Golden-set validation включает 2 281 размеченную пару: 1 960 оценок 1, 181 оценку 2 и 140 оценок 3.

## 3. Ручная проверка выдачи до метрик

Выбираем три validation-запроса с общим seed и показываем top-5 обоих методов по полному каталогу оценки.

In [4]:
manual_queries = golden_validation[['query_id', 'query_text']].drop_duplicates().sample(n=3, random_state=SEED).sort_values('query_id')
manual_results = []
for row in manual_queries.itertuples(index=False):
    for method, search in [('lexical', lexical.search_lexical), ('semantic', semantic.search_semantic)]:
        result = search(row.query_text, 5).copy()
        result.insert(0, 'method', method)
        result.insert(0, 'query_text', row.query_text)
        result.insert(0, 'query_id', row.query_id)
        manual_results.append(result)
manual_top5 = pd.concat(manual_results, ignore_index=True)
display(manual_top5)


,query_id,query_text,method,rank,imt_id,score,imt_name,subj_name,description
0,SYN_000209,брюки cop copine,lexical,1,10046272,0.688557,Брюки,Брюки,Брюки женские.
1,SYN_000209,брюки cop copine,lexical,2,10165757,0.643289,Брюки,Брюки,Брюки легкие
2,SYN_000209,брюки cop copine,lexical,3,10094352,0.535770,Брюки,Брюки,Базовые брюки прямого кроя прекрасны для офисн...
3,SYN_000209,брюки cop copine,lexical,4,10029153,0.523339,"Брюки ""Дженифер"" палаццо/прямые/широкие",Брюки,"Широкие брюки палаццо ""Дженифер"" со складками...."
4,SYN_000209,брюки cop copine,lexical,5,10105474,0.498228,Шорты,Шорты,Стильные шорты пудрово-розового оттенка из кач...
5,SYN_000209,брюки cop copine,semantic,1,10056719,0.605650,Кофта,Кофты,Кофта женская.
6,SYN_000209,брюки cop copine,semantic,2,10056715,0.587552,Джемпер,Джемперы,Джемпер женский.
7,SYN_000209,брюки cop copine,semantic,3,10148244,0.569458,Блузка женская,Блузки,Блузка-топ с принтом.
8,SYN_000209,брюки cop copine,semantic,4,10164885,0.565195,"Бомбер, бомбер женский, бомбер женский на осень",Бомберы,Великолепная куртка-бомбер из теплой и приятно...
9,SYN_000209,брюки cop copine,semantic,5,10085281,0.563829,Джемпер,Джемперы,Джемпер мужской.


**Наблюдение по `manual_top5`.** До агрегирования метрик обе системы показали выдачу для трёх фиксированных validation-запросов по полному каталогу. Эти примеры используются только как качественная проверка; выбор baseline сделан по `metrics_summary` ниже.

## 4. Проверка метрик на контролируемых сценариях

Перед оценкой реальных запросов подтверждаем формулы на идеальном и обратном ранжировании, неоценённом товаре, недоступном релевантном товаре и отсутствии релевантных результатов.

In [5]:
from calmfruits.evaluation import _query_metrics

perfect = _query_metrics([1, 2, 3], {1: 3, 2: 2}, {1, 2, 3})
reversed_rank = _query_metrics([3, 2, 1], {1: 3, 2: 2}, {1, 2, 3})
unrated = _query_metrics([3, 1, 2], {1: 3, 2: 2}, {1, 2, 3})
unreachable = _query_metrics([2, 3], {1: 3, 2: 2}, {2, 3})
no_relevant = _query_metrics([1, 2], {1: 0}, {1, 2})
assert perfect['ndcg_at_3'] == 1 and perfect['mrr'] == 1
assert reversed_rank['ndcg_at_3'] < perfect['ndcg_at_3'] and reversed_rank['mrr'] < perfect['mrr']
assert unrated['precision_at_1'] == 0 and unrated['mrr'] == .5
assert unreachable['reachable_relevant_share'] == .5 and unreachable['ndcg_upper_bound_at_10'] < 1
assert no_relevant['recall_at_10'] == 0 and no_relevant['ndcg_at_10'] == 0
metric_unit_cases = pd.DataFrame([perfect, reversed_rank, unrated, unreachable, no_relevant], index=['perfect', 'reversed', 'unrated_first', 'unreachable_relevant', 'no_relevant'])
display(metric_unit_cases[['precision_at_1', 'recall_at_1', 'mrr', 'ndcg_at_3', 'reachable_relevant_share', 'ndcg_upper_bound_at_10']])


,precision_at_1,recall_at_1,mrr,ndcg_at_3,reachable_relevant_share,ndcg_upper_bound_at_10
perfect,1.0,0.5,1.0,1.000000,1.0,1.000000
reversed,0.0,0.0,0.5,0.606423,1.0,1.000000
unrated_first,0.0,0.0,0.5,0.665315,1.0,1.000000
unreachable_relevant,1.0,0.5,1.0,0.337352,0.5,0.337352
no_relevant,0.0,0.0,0.0,0.000000,0.0,0.000000


**Вывод по `metric_unit_cases`.** Ручные сценарии подтверждают направление изменений метрик и обработку отсутствующей разметки до запуска на реальных запросах.

## 5. Оценка обоих поисков по одному golden-set

Каждый запрос ищет по всему общему каталогу оценки. Неоценённые карточки получают relevance 0; denominator Recall и IDCG сохраняют исходную разметку, включая релевантные товары вне каталога.

In [6]:
lexical_per_query, lexical_top10 = evaluate_search(lexical.search_lexical, golden_validation, evaluation_catalog.imt_id, 'lexical')
semantic_per_query, semantic_top10 = evaluate_search(semantic.search_semantic, golden_validation, evaluation_catalog.imt_id, 'semantic')
per_query_metrics = pd.concat([lexical_per_query, semantic_per_query], ignore_index=True)
top10_results = pd.concat([lexical_top10, semantic_top10], ignore_index=True)
metrics_summary = summarize_metrics(per_query_metrics)
catalog_coverage = per_query_metrics.groupby('method')[['relevant_items', 'reachable_relevant_items', 'reachable_relevant_share', 'ndcg_upper_bound_at_10']].mean().reset_index()
assert per_query_metrics.groupby('method').query_id.nunique().eq(golden_validation.query_id.nunique()).all()
per_query_metrics.to_csv(EVALUATION_DIR / 'per_query_metrics.csv', index=False)
top10_results.to_parquet(EVALUATION_DIR / 'top10_results.parquet', index=False)
metrics_summary.to_csv(REPORT_DIR / 'baseline_metrics.csv', index=False)
catalog_coverage.to_csv(REPORT_DIR / 'catalog_reachability.csv', index=False)
display(metrics_summary)
display(catalog_coverage)


,method,mrr,precision_at_1,recall_at_1,hit_rate_at_1,ndcg_at_1,precision_at_3,recall_at_3,hit_rate_at_3,ndcg_at_3,precision_at_5,recall_at_5,hit_rate_at_5,ndcg_at_5,precision_at_10,recall_at_10,hit_rate_at_10,ndcg_at_10
0,lexical,0.376057,0.250000,0.137608,0.250000,0.267347,0.202381,0.286913,0.442857,0.367170,0.167143,0.377675,0.514286,0.407521,0.124286,0.529084,0.628571,0.436972
1,semantic,0.214000,0.142857,0.064478,0.142857,0.137755,0.104762,0.137437,0.250000,0.171983,0.091429,0.183232,0.300000,0.183582,0.062857,0.236531,0.328571,0.185665


,method,relevant_items,reachable_relevant_items,reachable_relevant_share,ndcg_upper_bound_at_10
0,lexical,2.292857,2.278571,0.997959,0.908128
1,semantic,2.292857,2.278571,0.997959,0.908128


**Вывод по `metrics_summary` и `catalog_coverage`.** Лексический baseline превосходит семантический по NDCG@10 (0,437 против 0,186), MRR (0,376 против 0,214) и HitRate@10 (0,629 против 0,329). Средняя доля доступных бинарно релевантных товаров равна 0,998, но средний достижимый NDCG@10 равен 0,908, поэтому абсолютные значения метрик учитывают исключённые из каталога оценки товары. На текущем baseline нет доказательств в пользу замены лексического поиска семантическим.

## 6. Расхождения и ошибки выдачи

Выбираем три запроса с наибольшей разницей NDCG@10 между подходами, затем показываем top-10 каждого метода и исходную релевантность.

In [7]:
method_pivot = per_query_metrics.pivot(index=['query_id', 'query_text'], columns='method', values='ndcg_at_10').reset_index()
method_pivot['ndcg_gap_semantic_minus_lexical'] = method_pivot.semantic - method_pivot.lexical
error_queries = method_pivot.reindex(method_pivot.ndcg_gap_semantic_minus_lexical.abs().sort_values(ascending=False).index).head(3)
error_top10 = top10_results.merge(error_queries[['query_id', 'ndcg_gap_semantic_minus_lexical']], on='query_id', how='inner')
error_labels = golden_validation.merge(error_queries[['query_id']], on='query_id', how='inner').sort_values(['query_id', 'relevance'], ascending=[True, False])
display(error_queries)
display(error_top10[['query_id', 'method', 'rank', 'imt_id', 'relevance', 'imt_name', 'subj_name', 'score']])
display(error_labels[['query_id', 'query_text', 'item_id', 'relevance']])


method,query_id,query_text,lexical,semantic,ndcg_gap_semantic_minus_lexical
24,SYN_000156,шубы искусственные с капюшоном,0.986784,0.0,-0.986784
113,SYN_000832,сабо с открытым мысом glamforever,0.938799,0.0,-0.938799
44,SYN_000303,классические кожаные топсайдеры,0.917319,0.0,-0.917319


,query_id,method,rank,imt_id,relevance,imt_name,subj_name,score
0,SYN_000156,lexical,1,10013887,3,Шубы искусственные,Шубы искусственные,0.421263
1,SYN_000156,lexical,2,10164672,1,Шуба из искусственного меха/чебурашка/Искусств...,Шубы искусственные,0.193721
2,SYN_000156,lexical,3,10040802,0,Шубы натуральные,Шубы натуральные,0.180642
3,SYN_000156,lexical,4,10133829,1,Пальто женское из экомеха/ Шуба/ шуба искусств...,Шубы искусственные,0.176607
4,SYN_000156,lexical,5,10133830,1,Куртка женская из экомеха/ Шуба/ шуба искусств...,Шубы искусственные,0.171056
5,SYN_000156,lexical,6,10040803,0,Шубы натуральные,Шубы натуральные,0.164138
6,SYN_000156,lexical,7,10040823,0,Шубы натуральные,Шубы натуральные,0.155952
7,SYN_000156,lexical,8,10040810,0,Шубы натуральные,Шубы натуральные,0.149541
8,SYN_000156,lexical,9,10040824,0,Шубы натуральные,Шубы натуральные,0.148088
9,SYN_000156,lexical,10,10040813,0,Шубы натуральные,Шубы натуральные,0.146868


,query_id,query_text,item_id,relevance
0,SYN_000156,шубы искусственные с капюшоном,10013887,3
1,SYN_000156,шубы искусственные с капюшоном,10133829,1
2,SYN_000156,шубы искусственные с капюшоном,10133830,1
3,SYN_000156,шубы искусственные с капюшоном,10164672,1
4,SYN_000303,классические кожаные топсайдеры,10038068,3
5,SYN_000303,классические кожаные топсайдеры,10038070,1
20,SYN_000832,сабо с открытым мысом glamforever,10154138,3
6,SYN_000832,сабо с открытым мысом glamforever,101032,1
7,SYN_000832,сабо с открытым мысом glamforever,10002815,1
8,SYN_000832,сабо с открытым мысом glamforever,10012526,1


**Вывод по `error_queries`, `error_top10` и `error_labels`.** Для «шубы искусственные с капюшоном», «сабо с открытым мысом glamforever» и «классические кожаные топсайдеры» лексический поиск ставит товар с relevance 3 на первое место, тогда как семантический не возвращает релевантный товар в top-10. Проверяемые гипотезы третьей недели: полный `product_text` с длинными описаниями размывает семантический вектор, а гибридная выдача может сохранить преимущество точных совпадений.

## 7. Чек-лист недели

Перед завершением этапа запустить тетради 02 и 03 в чистых kernels, убедиться, что все assertions выполнились, обновить workbook и провести независимое Astra-review. Test-разметку не использовать.